# Terra Signal: análise exploratória (Grupo 1)

Perguntas para (1) achar inconsistências nos dados e (2) orientar a construção do modelo de churn.

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- Dados: `history.csv` (com `Churn`) e `inference.csv` (sem `Churn`), na raiz do repositório.
- Roda com pandas e scikit-learn, sem Spark. Funciona localmente e no Databricks (Git folder).

## Setup

In [1]:
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 300)

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
inf_raw = pd.read_csv(DATA_DIR + "inference.csv")
print("history:", hist_raw.shape, "| inference:", inf_raw.shape)

history: (5634, 23) | inference: (1409, 22)


**Constantes e limpeza.** A Parte 1 inspeciona os dados brutos (`hist_raw`, `inf_raw`).
A função `prep` aplica as correções que a própria Parte 1 justifica e gera `hist` e `inf`, usados nas perguntas que dependem de dados limpos.

In [2]:
ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

# Feedback dizendo que o cliente JÁ cancelou
CANCEL_NARROW = r"decided to (?:churn|cancel|leave|switch)|have churned|canceled|cancelled|ended my|terminated"
CANCEL_BROAD = (
    r"decided to (?:churn|cancel|leave|switch|discontinue)|had to (?:churn|cancel|leave|switch|discontinue)"
    r"|needed to cancel|have churned|i churned|canceled|cancelled|ended my|terminated"
    r"|decision to (?:churn|cancel|leave)|chose to (?:churn|cancel|leave)"
)
# Tom negativo no feedback
NEGATIVE = (
    r"unfortunately|frustrat|disappoint|not satisfied|unsatisf|dissatisf|unreliable|slow|outage|expensive|steep|too high|poor"
    r"|considering (?:switching|leaving|other)|thinking (?:about|of) (?:switching|leaving)"
)


def prep(d):
    """Limpeza: padroniza PhoneService, recupera tenure e imputa TotalCharges."""
    d = d.copy()
    d["PhoneService"] = d["PhoneService"].str.capitalize()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    d["TotalCharges_imp"] = d["TotalCharges"].fillna(d["MonthlyCharges"] * d["tenure_num"])
    d.loc[d["tenure_num"] == 0, "TotalCharges_imp"] = 0
    return d


hist = prep(hist_raw)
inf = prep(inf_raw)
y = (hist["Churn"] == "Yes").astype(int)

# Parte 1: inconsistências nos dados

## A. Formato e integridade

### 1. O `customerID` é único e não se repete entre history e inference?

In [3]:
for name, d in [("history", hist_raw), ("inference", inf_raw)]:
    print(name, "| IDs duplicados:", d["customerID"].duplicated().sum())
print("IDs em comum entre as bases:", len(set(hist_raw["customerID"]) & set(inf_raw["customerID"])))

history | IDs duplicados: 0
inference | IDs duplicados: 0
IDs em comum entre as bases: 0


**Resposta:** Sim. Não há IDs duplicados em nenhuma das bases, e as duas não têm clientes em comum.

### 2. Todos os IDs seguem o mesmo padrão (`NNNN-XXXXX`)?

In [4]:
for name, d in [("history", hist_raw), ("inference", inf_raw)]:
    ok = d["customerID"].str.fullmatch(r"\d{4}-[A-Z]{5}")
    print(f"{name}: {ok.mean():.1%} no padrão")

history: 100.0% no padrão
inference: 100.0% no padrão


**Resposta:** Sim, 100% nas duas bases.

### 3. As colunas têm o tipo certo?

In [5]:
print(hist_raw.dtypes)
t = pd.to_numeric(hist_raw["tenure"], errors="coerce")
print("\nValores não numéricos em tenure (history):", hist_raw.loc[t.isna(), "tenure"].value_counts().to_dict())
print("Valores não numéricos em tenure (inference):",
      inf_raw.loc[pd.to_numeric(inf_raw["tenure"], errors="coerce").isna(), "tenure"].value_counts().to_dict())

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                  str
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
CustomerFeedback        str
MonthlyIncome         int64
dtype: object

Valores não numéricos em tenure (history): {'unknown': 19}
Valores não numéricos em tenure (inference): {'unknown': 4}


**Resposta:** Não. `tenure` é lido como texto porque tem o valor `"unknown"` em 19 linhas do history e 4 do inference. As outras colunas têm o tipo esperado.

### 4. Há valores faltantes disfarçados (`""`, espaços, `"N/A"`, `"unknown"`) além dos nulos reais?

In [6]:
NA_LIKE = ["na", "n/a", "null", "none", "nan", "unknown", "?", "-"]
for name, d in [("history", hist_raw), ("inference", inf_raw)]:
    print(f"== {name}")
    print("Nulos reais:", d.isna().sum()[d.isna().sum() > 0].to_dict())
    for c in d.select_dtypes(exclude="number").columns:
        s = d[c].dropna().astype(str)
        blank = (s.str.strip() == "").sum()
        spaces = (s != s.str.strip()).sum()
        na_like = s.str.lower().isin(NA_LIKE).sum()
        if blank or spaces or na_like:
            print(f"  {c}: vazios={blank} espaços={spaces} na-like={na_like}")

== history
Nulos reais: {'TotalCharges': 278}
  tenure: vazios=0 espaços=0 na-like=19
== inference
Nulos reais: {'TotalCharges': 64}
  tenure: vazios=0 espaços=0 na-like=4


**Resposta:** Os únicos nulos reais estão em `TotalCharges` (278 no history e 64 no inference). O único faltante disfarçado é o `"unknown"` de `tenure`. Não há vazios, espaços sobrando nem `"N/A"`.

### 5. Alguma coluna categórica tem variação de grafia (maiúsculas ou espaços)?

In [7]:
for name, d in [("history", hist_raw), ("inference", inf_raw)]:
    for c in d.columns:
        if d[c].dtype == object or str(d[c].dtype) == "str":
            if d[c].nunique() < 10 and d[c].str.strip().str.lower().nunique() != d[c].nunique():
                print(name, c, d[c].value_counts().to_dict())

history PhoneService {'Yes': 5041, 'No': 523, 'no': 40, 'yes': 30}


inference PhoneService {'Yes': 1277, 'No': 109, 'yes': 13, 'no': 10}


**Resposta:** Só `PhoneService`, que mistura `Yes/No` com `yes/no`: 70 linhas no history e 23 no inference. A função `prep` corrige isso.

### 6. Existem linhas duplicadas se ignorarmos o ID?

In [8]:
for name, d in [("history", hist_raw), ("inference", inf_raw)]:
    print(name, "| duplicadas sem ID:", d.drop(columns="customerID").duplicated().sum(),
          "| sem ID e sem feedback:", d.drop(columns=["customerID", "CustomerFeedback"]).duplicated().sum())

history | duplicadas sem ID: 0 | sem ID e sem feedback: 0
inference | duplicadas sem ID: 0 | sem ID e sem feedback: 0


**Resposta:** Não. Mesmo ignorando o ID e o feedback, não há linhas repetidas.

## B. Regras de negócio entre colunas

### 7. `No phone service` (em `MultipleLines`) aparece só quando `PhoneService = No`?

In [9]:
for name, d in [("history", hist), ("inference", inf)]:
    print(name)
    print(pd.crosstab(d["PhoneService"], d["MultipleLines"]), "\n")

history
MultipleLines    No  No phone service   Yes
PhoneService                               
No                0               563     0
Yes            2704                 0  2367 

inference
MultipleLines   No  No phone service  Yes
PhoneService                             
No               0               119    0
Yes            686                 0  604 



**Resposta:** Sim, sem nenhuma violação nas duas bases. A regra confirmada pela TS se cumpre.

### 8. `No internet service` nos 6 serviços adicionais aparece só quando `InternetService = No`? Todo cliente tem telefone ou internet?

In [10]:
for name, d in [("history", hist), ("inference", inf)]:
    viol = sum(((d[c] == "No internet service") != (d["InternetService"] == "No")).sum() for c in ADDONS)
    sem_nada = ((d["PhoneService"] == "No") & (d["InternetService"] == "No")).sum()
    print(f"{name}: violações = {viol} | clientes sem telefone e sem internet = {sem_nada}")

history: violações = 0 | clientes sem telefone e sem internet = 0
inference: violações = 0 | clientes sem telefone e sem internet = 0


**Resposta:** Sim nos dois casos: nenhuma violação, e todo cliente tem pelo menos telefone ou internet.

### 9. `TotalCharges` é aproximadamente `MonthlyCharges × tenure`?

In [11]:
for name, d in [("history", hist), ("inference", inf)]:
    ratio = d["TotalCharges"] / (d["MonthlyCharges"] * d["tenure_num"])
    ratio = ratio.replace(np.inf, np.nan)
    print(name, ratio.describe(percentiles=[.05, .5, .95]).round(3).to_dict())
    print("  TotalCharges < MonthlyCharges:", (d["TotalCharges"] < d["MonthlyCharges"] - 0.01).sum())

history {'count': 5356.0, 'mean': 1.0, 'std': 0.051, 'min': 0.689, '5%': 0.925, '50%': 1.0, '95%': 1.075, 'max': 1.495}
  TotalCharges < MonthlyCharges: 0
inference {'count': 1345.0, 'mean': 1.002, 'std': 0.052, 'min': 0.765, '5%': 0.93, '50%': 1.0, '95%': 1.078, 'max': 1.573}
  TotalCharges < MonthlyCharges: 0


**Resposta:** Sim. A mediana da razão é 1,00 e 90% dos casos ficam entre 0,93 e 1,08. Nenhum cliente tem `TotalCharges` menor que a mensalidade. Isso justifica imputar os nulos com `Monthly × tenure`.

### 10. Quem são os clientes com diferença acima de 15% entre `TotalCharges` e `MonthlyCharges × tenure`? É reajuste, desconto ou erro?

In [12]:
for name, d in [("history", hist), ("inference", inf)]:
    t = pd.to_numeric(d["tenure"], errors="coerce")
    ratio = d["TotalCharges"] / (d["MonthlyCharges"] * t)
    out = (ratio - 1).abs() > 0.15
    print(f"{name}: {out.sum()} linhas | tenure dessas linhas:", t[out].describe().round(1)[["min", "50%", "max"]].to_dict())
    print("   % fora por faixa de tenure:", out.groupby(pd.cut(t, [0, 1, 3, 6, 12, 72])).mean().round(3).to_dict())
    if "Churn" in d:
        print("   taxa de churn nessas linhas:", round((d.loc[out, "Churn"] == "Yes").mean(), 3))

history: 113 linhas | tenure dessas linhas: {'min': 2.0, '50%': 4.0, 'max': 27.0}
   % fora por faixa de tenure: {Interval(0, 1, closed='right'): 0.0, Interval(1, 3, closed='right'): 0.149, Interval(3, 6, closed='right'): 0.076, Interval(6, 12, closed='right'): 0.037, Interval(12, 72, closed='right'): 0.004}
   taxa de churn nessas linhas: 0.23
inference: 28 linhas | tenure dessas linhas: {'min': 2.0, '50%': 4.5, 'max': 25.0}
   % fora por faixa de tenure: {Interval(0, 1, closed='right'): 0.0, Interval(1, 3, closed='right'): 0.114, Interval(3, 6, closed='right'): 0.081, Interval(6, 12, closed='right'): 0.07, Interval(12, 72, closed='right'): 0.003}


**Resposta:** São 113 linhas no history e 28 no inference, quase todas com tenure de 2 a 8 meses. Com poucos meses, um reajuste ou uma promoção pesa muito na razão. Acima de 12 meses, só 0,4% das linhas ficam fora. A taxa de churn desse grupo (23%) é normal. **Não parece erro.**

### 11. A `MonthlyCharges` é coerente com os serviços contratados (existe uma tabela de preços)?

In [13]:
def price_features(d):
    X = pd.DataFrame({
        "phone": (d["PhoneService"] == "Yes").astype(int),
        "multi_lines": (d["MultipleLines"] == "Yes").astype(int),
        "dsl": (d["InternetService"] == "DSL").astype(int),
        "fiber": (d["InternetService"] == "Fiber optic").astype(int),
    })
    for c in ADDONS:
        X[c] = (d[c] == "Yes").astype(int)
    return X

price_model = LinearRegression().fit(price_features(hist), hist["MonthlyCharges"])
hist["price_residual"] = hist["MonthlyCharges"] - price_model.predict(price_features(hist))
inf["price_residual"] = inf["MonthlyCharges"] - price_model.predict(price_features(inf))

print("R²:", round(price_model.score(price_features(hist), hist["MonthlyCharges"]), 4))
print("Intercepto:", round(price_model.intercept_, 2))
print("Preço estimado de cada serviço:", dict(zip(price_features(hist).columns, price_model.coef_.round(2))))
print("Resíduo — quantis:", hist["price_residual"].quantile([.01, .5, .99]).round(2).to_dict(),
      "| |resíduo| > 10:", (hist["price_residual"].abs() > 10).sum())

R²: 0.9989
Intercepto: -0.1


Preço estimado de cada serviço: {'phone': np.float64(20.05), 'multi_lines': np.float64(5.02), 'dsl': np.float64(25.06), 'fiber': np.float64(50.01), 'OnlineSecurity': np.float64(5.03), 'OnlineBackup': np.float64(5.0), 'DeviceProtection': np.float64(5.01), 'TechSupport': np.float64(4.99), 'StreamingTV': np.float64(9.99), 'StreamingMovies': np.float64(9.95)}
Resíduo — quantis: {0.01: -2.52, 0.5: -0.0, 0.99: 2.61} | |resíduo| > 10: 0


**Resposta:** **Sim, a tabela de preços é quase exata (R² = 0,999):** telefone R$ 20, linha extra +R$ 5, DSL R$ 25, **fibra R$ 50**, segurança, backup, proteção e suporte +R$ 5 cada, cada streaming +R$ 10. O resíduo fica em ±R$ 2,50, e nenhum cliente está fora da tabela.

### 12. `MonthlyIncome` é plausível e tem relação com o resto?

In [14]:
d = hist
print(d["MonthlyIncome"].describe().round(0).to_dict())
print("Histograma (10 faixas iguais):", np.histogram(d["MonthlyIncome"], bins=10)[0].tolist())
print("Renda <= R$ 500:", (d["MonthlyIncome"] <= 500).sum(), "| renda < mensalidade:", (d["MonthlyIncome"] < d["MonthlyCharges"]).sum())
print("Correlação renda × mensalidade:", round(d["MonthlyIncome"].corr(d["MonthlyCharges"]), 3))
print("Renda média por SeniorCitizen:", d.groupby("SeniorCitizen")["MonthlyIncome"].mean().round(0).to_dict())
print("Renda média por Dependents:", d.groupby("Dependents")["MonthlyIncome"].mean().round(0).to_dict())
print("Churn por quartil de renda:", y.groupby(pd.qcut(d["MonthlyIncome"], 4)).mean().round(3).to_dict())

{'count': 5634.0, 'mean': 5105.0, 'std': 2839.0, 'min': 100.0, '25%': 2694.0, '50%': 5080.0, '75%': 7575.0, 'max': 9993.0}
Histograma (10 faixas iguais): [532, 549, 520, 609, 588, 569, 544, 573, 557, 593]
Renda <= R$ 500: 190 | renda < mensalidade: 1
Correlação renda × mensalidade: 0.011
Renda média por SeniorCitizen: {0: 5106.0, 1: 5097.0}
Renda média por Dependents: {'No': 5114.0, 'Yes': 5083.0}
Churn por quartil de renda: {Interval(99.999, 2693.75, closed='right'): 0.26, Interval(2693.75, 5079.5, closed='right'): 0.261, Interval(5079.5, 7574.75, closed='right'): 0.278, Interval(7574.75, 9993.0, closed='right'): 0.268}


**Resposta:** **Parece ruído aleatório.** A distribuição é uniforme de R$ 100 a R$ 10.000, a correlação com a mensalidade é ≈ 0, a média é a mesma para idosos e não idosos e para quem tem ou não dependentes, e o churn fica entre 26% e 28% em todos os quartis. 190 clientes têm renda de até R$ 500. Candidata a ser removida (ver pergunta 31).

## C. Texto (`CustomerFeedback`) contra as colunas

### 13. O valor em `$` citado no feedback é igual à `MonthlyCharges`?

In [15]:
for name, d in [("history", hist), ("inference", inf)]:
    val = d["CustomerFeedback"].str.extract(r"\$\s?([0-9]+(?:\.[0-9]+)?)")[0].astype(float)
    print(f"{name}: {val.notna().mean():.1%} citam um valor | iguais à MonthlyCharges: "
          f"{(np.isclose(val, d['MonthlyCharges']) & val.notna()).sum()} de {val.notna().sum()}")

history: 90.1% citam um valor | iguais à MonthlyCharges: 5077 de 5077
inference: 89.9% citam um valor | iguais à MonthlyCharges: 1266 de 1266


**Resposta:** Sim, em 100% dos casos. Cerca de 90% dos textos citam o valor, e ele é sempre igual à `MonthlyCharges`. O texto usa `$`, mas a TS mandou considerar R$.

### 14. Os meses citados no texto batem com `tenure`? Dá para recuperar o `"unknown"`?

In [16]:
for name, d in [("history", hist), ("inference", inf)]:
    t = pd.to_numeric(d["tenure"], errors="coerce")
    months = d["CustomerFeedback"].str.extract(r"(?<![\d.$])(\d+)\s+months?\b", flags=re.I)[0].astype(float)
    mismatch = months.notna() & t.notna() & (months != t)
    print(f"{name}: {months.notna().sum()} citam meses | não batem: {mismatch.sum()} | diferenças:",
          (months - t)[mismatch].value_counts().to_dict())
    print(f"   tenure 'unknown' com meses no texto: {(t.isna() & months.notna()).sum()} de {t.isna().sum()}")
    print(f"   'unknown' que continuam sem tenure depois do prep: {d['tenure_num'].isna().sum()}")

history: 3084 citam meses | não batem: 1 | diferenças: {-60.0: 1}
   tenure 'unknown' com meses no texto: 8 de 19
   'unknown' que continuam sem tenure depois do prep: 1


inference: 763 citam meses | não batem: 3 | diferenças: {-60.0: 2, 1.0: 1}
   tenure 'unknown' com meses no texto: 2 de 4
   'unknown' que continuam sem tenure depois do prep: 0


**Resposta:** Sim. Das 3.084 menções no history, só 1 não bate, e das 763 no inference, só 3. Combinando `TotalCharges / MonthlyCharges` (usado no `prep`) com o texto, quase todos os `"unknown"` são recuperados: sobra 1 caso no history.

### 15. O tipo de internet citado no texto bate com `InternetService`?

In [17]:
fb = hist["CustomerFeedback"]
print("Menciona fibra:\n", pd.crosstab(fb.str.contains(r"fiber|fibre", case=False), hist["InternetService"]), "\n")
print("Menciona DSL:\n", pd.crosstab(fb.str.contains(r"\bDSL\b", case=False), hist["InternetService"]), "\n")
no_int = fb.str.contains(r"(?:no|without|don't have|do not have|lack of)\s+(?:an?\s+)?internet", case=False)
print("Menciona não ter internet:\n", pd.crosstab(no_int, hist["InternetService"]))

Menciona fibra:
 InternetService    DSL  Fiber optic    No
CustomerFeedback                         
False             1924            6  1233
True                 0         2471     0 

Menciona DSL:
 InternetService    DSL  Fiber optic    No
CustomerFeedback                         
False               25         2477  1233
True              1899            0     0 



Menciona não ter internet:
 InternetService    DSL  Fiber optic   No
CustomerFeedback                        
False             1924         2477  398
True                 0            0  835


**Resposta:** Sim, praticamente 100%. Ninguém com DSL cita fibra, nenhum cliente de fibra cita DSL, e quem diz não ter internet de fato não tem.

### 16. A forma de pagamento citada no texto bate com `PaymentMethod`?

In [18]:
patterns = {
    "Electronic check": r"electronic check|e-check",
    "Mailed check": r"mail(?:ed|ing)?\s+(?:in\s+)?(?:a\s+)?check|check by mail|paper check",
    "Bank transfer (automatic)": r"bank transfer",
    "Credit card (automatic)": r"credit card",
}
fb = hist["CustomerFeedback"]
for method, pat in patterns.items():
    m = fb.str.contains(pat, case=False)
    print(f"{method}: menções={m.sum()} | batem={(m & (hist['PaymentMethod'] == method)).sum()} | não batem={(m & (hist['PaymentMethod'] != method)).sum()}")

print("\nExemplos de 'credit card' citado por quem não paga com cartão:")
cc = hist[fb.str.contains("credit card", case=False) & (hist["PaymentMethod"] != "Credit card (automatic)")]
for _, r in cc.head(3).iterrows():
    print(" -", r["PaymentMethod"], "|", re.findall(r"[^.]*credit card[^.]*", r["CustomerFeedback"], re.I)[0].strip())

Electronic check: menções=1848 | batem=1848 | não batem=0


Mailed check: menções=1276 | batem=1276 | não batem=0
Bank transfer (automatic): menções=1225 | batem=1224 | não batem=1
Credit card (automatic): menções=1178 | batem=1166 | não batem=12

Exemplos de 'credit card' citado por quem não paga com cartão:
 - Electronic check | I also found the payment method of electronic check to be inconvenient and would have preferred more options like credit card or automatic billing
 - Electronic check | Additionally, the payment method of electronic check was not convenient for me as I prefer using a credit card
 - Electronic check | Additionally, I prefer to pay my bills using a credit card rather than electronic check, so that was another factor that led me to move on


**Resposta:** Sim. As poucas divergências (13 no history) são clientes de cheque eletrônico dizendo que *prefeririam* pagar com cartão. Não são erros.

### 17. Quem são os clientes com churn cujo texto não diz que cancelaram?

In [19]:
fb = hist["CustomerFeedback"]
narrow = fb.str.contains(CANCEL_NARROW, case=False)
broad = fb.str.contains(CANCEL_BROAD, case=False)
print("Churn sem texto de cancelamento — lista curta:", ((y == 1) & ~narrow).sum(), "| lista ampla:", ((y == 1) & ~broad).sum())
rest = hist[(y == 1) & ~broad]
print("Desses, com tom negativo:", f"{rest['CustomerFeedback'].str.contains(NEGATIVE, case=False).mean():.0%}")
for txt in rest["CustomerFeedback"].sample(3, random_state=0):
    print(" -", txt[:250])

Churn sem texto de cancelamento — lista curta: 146 | lista ampla: 29
Desses, com tom negativo: 93%
 - I recently switched to this fiber optic internet service provider three months ago on a month-to-month contract. The monthly charges of $74.4 seemed reasonable at first. However, I have experienced constant disruptions in my service and slow speeds. 
 - I recently switched to this internet service provider and unfortunately, I will be churning after only one month. The monthly charges of $74.50 were reasonable for fiber optic service, but I had issues with the reliability of the connection and slow 
 - I have been a customer with this company for over four and a half years and I must say I am extremely disappointed with the service I have received. Despite being on a one-year contract, I have experienced frequent outages and slow internet speeds. T


**Resposta:** Com a lista curta de expressões sobravam 146. Com a lista ampla ("had to cancel", "needed to cancel" etc.), sobram só **29**, e mesmo esses têm tom negativo. Quase todo churner diz no texto que saiu.

## D. History contra inference

### 18. As distribuições das variáveis são parecidas nas duas bases (drift)?

In [20]:
for c in ["Contract", "InternetService", "PaymentMethod", "SeniorCitizen", "PaperlessBilling", "TechSupport"]:
    diff = (hist[c].value_counts(normalize=True) - inf[c].value_counts(normalize=True)).abs().max()
    print(f"{c}: maior diferença de proporção = {diff:.3f}")
for c in ["tenure_num", "MonthlyCharges", "TotalCharges_imp", "MonthlyIncome"]:
    print(f"{c}: média history={hist[c].mean():.1f} | inference={inf[c].mean():.1f}")

Contract: maior diferença de proporção = 0.009
InternetService: maior diferença de proporção = 0.011
PaymentMethod: maior diferença de proporção = 0.015
SeniorCitizen: maior diferença de proporção = 0.013
PaperlessBilling: maior diferença de proporção = 0.008
TechSupport: maior diferença de proporção = 0.022
tenure_num: média history=32.3 | inference=32.5
MonthlyCharges: média history=64.6 | inference=65.4
TotalCharges_imp: média history=2272.1 | inference=2308.5
MonthlyIncome: média history=5104.9 | inference=5146.2


**Resposta:** Sim, são parecidas. As proporções diferem no máximo 2,2 p.p. e as médias são praticamente iguais. Não há drift.

### 19. A proporção de nulos e de textos de cancelamento é parecida nas duas bases?

In [21]:
for name, d in [("history", hist_raw), ("inference", inf_raw)]:
    print(f"{name}: TotalCharges nulo = {d['TotalCharges'].isna().mean():.1%} | "
          f"texto de cancelamento = {d['CustomerFeedback'].str.contains(CANCEL_BROAD, case=False).mean():.1%}")

history: TotalCharges nulo = 4.9% | texto de cancelamento = 26.2%


inference: TotalCharges nulo = 4.5% | texto de cancelamento = 25.6%


**Resposta:** Sim: 4,9% contra 4,5% de nulos e 26,2% contra 25,6% de textos de cancelamento. É coerente com uma taxa de churn semelhante no inference, de cerca de 26%.

# Parte 2: construção do modelo

## E. O alvo

### 20. Qual é a taxa de churn?

In [22]:
print(hist["Churn"].value_counts(), "\n")
print(f"Taxa de churn: {y.mean():.1%}")

Churn
No     4130
Yes    1504
Name: count, dtype: int64 

Taxa de churn: 26.7%


**Resposta:** 26,7% (1.504 de 5.634). As classes são desbalanceadas: acurácia sozinha engana, já que prever sempre "No" daria 73%.

### 21. Qual métrica a TS usa para avaliar o `prediction.csv`?

**Resposta:** Os dados não respondem isso. **A pergunta precisa ir para a TS.** A resposta define se otimizamos acurácia, F1, recall ou AUC, e se entregamos `Yes/No` ou uma probabilidade.

## F. Fatores de churn

### 22. Qual é a taxa de churn por variável?

In [23]:
for c in ["Contract", "InternetService", "PaymentMethod", "SeniorCitizen", "Partner", "Dependents", "PaperlessBilling", "TechSupport"]:
    print(c, y.groupby(hist[c]).mean().round(3).sort_values(ascending=False).to_dict())
print("tenure", y.groupby(pd.cut(hist["tenure_num"], [-1, 6, 12, 24, 48, 72])).mean().round(3).to_dict())
print("MonthlyCharges", y.groupby(pd.qcut(hist["MonthlyCharges"], 4)).mean().round(3).to_dict())

Contract {'Month-to-month': 0.427, 'One year': 0.116, 'Two year': 0.033}
InternetService {'Fiber optic': 0.421, 'DSL': 0.194, 'No': 0.072}
PaymentMethod {'Electronic check': 0.451, 'Mailed check': 0.194, 'Bank transfer (automatic)': 0.172, 'Credit card (automatic)': 0.156}
SeniorCitizen {1: 0.423, 0: 0.236}
Partner {'No': 0.329, 'Yes': 0.201}
Dependents {'No': 0.316, 'Yes': 0.153}
PaperlessBilling {'Yes': 0.337, 'No': 0.166}
TechSupport {'No': 0.418, 'Yes': 0.153, 'No internet service': 0.072}
tenure {Interval(-1, 6, closed='right'): 0.523, Interval(6, 12, closed='right'): 0.354, Interval(12, 24, closed='right'): 0.286, Interval(24, 48, closed='right'): 0.218, Interval(48, 72, closed='right'): 0.095}
MonthlyCharges {Interval(18.398999999999997, 35.162, closed='right'): 0.11, Interval(35.162, 70.3, closed='right'): 0.246, Interval(70.3, 89.8, closed='right'): 0.373, Interval(89.8, 118.65, closed='right'): 0.338}


**Resposta:** Os maiores riscos são contrato mensal (42,7%), tenure de até 6 meses (52,3%), fibra (42,1%), cheque eletrônico (45,1%), idosos (42,3%), cliente sem TechSupport (41,8%) e mensalidade de R$ 70–90 (37,3%). Os menores são contrato de 2 anos (3,3%) e sem internet (7,2%).

### 23. Como as variáveis interagem?

In [24]:
print("Contrato × tenure\n", y.groupby([hist["Contract"], pd.cut(hist["tenure_num"], [-1, 6, 12, 24, 72])]).mean().unstack().round(3), "\n")
print("Internet × TechSupport\n", y.groupby([hist["InternetService"], hist["TechSupport"]]).mean().unstack().round(3), "\n")
print("Pagamento × fatura digital\n", y.groupby([hist["PaymentMethod"], hist["PaperlessBilling"]]).mean().unstack().round(3), "\n")
print("Idoso × internet\n", y.groupby([hist["SeniorCitizen"], hist["InternetService"]]).mean().unstack().round(3))

Contrato × tenure
 tenure_num      (-1, 6]  (6, 12]  (12, 24]  (24, 72]
Contract                                            
Month-to-month    0.546    0.416     0.383     0.315
One year          0.118    0.111     0.076     0.122
Two year          0.000    0.000     0.000     0.036 

Internet × TechSupport
 TechSupport         No  No internet service    Yes
InternetService                                   
DSL              0.283                  NaN  0.098
Fiber optic      0.493                  NaN  0.229
No                 NaN                0.072    NaN 

Pagamento × fatura digital
 PaperlessBilling              No    Yes
PaymentMethod                          
Bank transfer (automatic)  0.108  0.220
Credit card (automatic)    0.101  0.195
Electronic check           0.341  0.490
Mailed check               0.138  0.276 

Idoso × internet
 InternetService    DSL  Fiber optic     No
SeniorCitizen                             
0                0.178        0.400  0.071
1               

**Resposta:** **Contrato mensal com até 6 meses: 54,6%.** Contrato de 2 anos: 0% no início e 3,6% depois. **Fibra sem TechSupport: 49,3%** (fibra com suporte: 22,9%). **Cheque eletrônico com fatura digital: 49,0%.** Idoso com fibra: 47,6%.

### 24. Ter mais serviços adicionais segura o cliente?

In [25]:
n_addons = sum((hist[c] == "Yes").astype(int) for c in ADDONS)
has_internet = hist["InternetService"] != "No"
print(y[has_internet].groupby(n_addons[has_internet]).agg(["mean", "size"]).round(3).T)

            0        1        2       3       4        5        6
mean    0.509    0.452    0.377    0.28    0.22    0.129    0.069
size  552.000  770.000  822.000  899.00  676.00  464.000  218.000


**Resposta:** **Muito.** Entre clientes com internet, o churn cai de forma monotônica: 50,9% sem serviços → 28,0% com 3 → 6,9% com 6.

### 25. Pagar acima do preço esperado aumenta o churn? (hipótese da TS de "conta alta para os serviços")

In [26]:
print(y.groupby(pd.qcut(hist["price_residual"], 5)).mean().round(3).to_dict())

{Interval(-4.243, -0.759, closed='right'): 0.282, Interval(-0.759, -0.228, closed='right'): 0.255, Interval(-0.228, 0.199, closed='right'): 0.251, Interval(0.199, 0.749, closed='right'): 0.261, Interval(0.749, 4.655, closed='right'): 0.285}


**Resposta:** **Não.** Ninguém paga acima da tabela (pergunta 11), e o churn fica entre 25% e 28% em todos os quintis do resíduo. O que pesa é a conta alta em si, puxada pela fibra, e não um sobrepreço.

### 26. A renda não explica o churn sozinha. E o peso da conta na renda (`MonthlyCharges / MonthlyIncome`)?

In [27]:
ratio = hist["MonthlyCharges"] / hist["MonthlyIncome"]
print(y.groupby(pd.qcut(ratio, 5)).mean().round(3).to_dict())

{Interval(0.0009499999999999999, 0.00611, closed='right'): 0.138, Interval(0.00611, 0.0106, closed='right'): 0.275, Interval(0.0106, 0.0156, closed='right'): 0.32, Interval(0.0156, 0.029, closed='right'): 0.303, Interval(0.029, 1.015, closed='right'): 0.299}


**Resposta:** Só o quintil mais baixo difere (13,8% contra cerca de 30%), e isso vem da mensalidade baixa, não da renda (ver pergunta 12). Não acrescenta informação.

### 27. O churn dos clientes novos (≤ 6 meses) se concentra em algum tipo de serviço ou pagamento?

In [28]:
new = hist["tenure_num"] <= 6
for c in ["InternetService", "PaymentMethod", "Contract"]:
    print(c, y[new].groupby(hist.loc[new, c]).agg(["mean", "size"]).round(3).to_dict("index"))

InternetService {'DSL': {'mean': 0.488, 'size': 408}, 'Fiber optic': {'mean': 0.735, 'size': 487}, 'No': {'mean': 0.211, 'size': 284}}
PaymentMethod {'Bank transfer (automatic)': {'mean': 0.484, 'size': 91}, 'Credit card (automatic)': {'mean': 0.454, 'size': 108}, 'Electronic check': {'mean': 0.664, 'size': 538}, 'Mailed check': {'mean': 0.378, 'size': 442}}
Contract {'Month-to-month': {'mean': 0.546, 'size': 1122}, 'One year': {'mean': 0.118, 'size': 34}, 'Two year': {'mean': 0.0, 'size': 23}}


**Resposta:** Sim. **Fibra: 73,5%** (DSL 48,8%, sem internet 21,1%). Cheque eletrônico: 66,4%. 95% dos clientes novos estão no contrato mensal.

## G. Feedback (NLP)

### 28. Quais dores aparecem nos feedbacks e quais estão mais ligadas ao churn?

In [29]:
THEMES = {
    "preço": r"expensive|too high|steep|overpriced|pricey|costly|high (?:monthly )?(?:charges|price|cost)|not worth|cheaper|better (?:deal|price|value)",
    "velocidade": r"\bslow|speed",
    "instabilidade": r"unreliab|outage|disconnect|interruption|downtime|connectivity issues|buffering|spotty|inconsistent",
    "pagamento": r"payment (?:issue|problem|process)|inconvenient|hassle|late fee|billing (?:issue|error|problem)|delay",
    "atendimento": r"customer service|customer support|tech(?:nical)? support|support team|rude|unhelpful",
    "contrato": r"contract|commitment|flexib",
    "mudança de vida": r"moving|moved|relocat|circumstances|financial",
}
fb = hist["CustomerFeedback"]
rows = []
for theme, pat in THEMES.items():
    m = fb.str.contains(pat, case=False)
    rows.append({"tema": theme, "% de todos": m.mean(), "% dos churners": m[y == 1].mean(),
                 "churn se menciona": y[m].mean(), "churn se não menciona": y[~m].mean()})
pd.DataFrame(rows).set_index("tema").round(3)

,% de todos,% dos churners,churn se menciona,churn se não menciona
tema,,,,
preço,0.114,0.427,1.000,0.173
velocidade,0.210,0.376,0.478,0.211
instabilidade,0.153,0.245,0.426,0.238
pagamento,0.249,0.305,0.327,0.247
atendimento,0.028,0.055,0.512,0.260
contrato,0.416,0.682,0.437,0.146
mudança de vida,0.015,0.057,1.000,0.256


**Resposta:** **Preço**: citado por 42,7% dos churners, e 100% de quem menciona cancelou. **Contrato/flexibilidade**: 68,2% dos churners. **Velocidade**: 37,6%. **Pagamento**: 30,5%. **Instabilidade**: 24,5%. **Atendimento**: 5,5%. **Mudança de vida**: 5,7%, com 100% de churn.

### 29. Clientes ativos reclamam no feedback? (o alvo ideal de retenção)

In [30]:
fb = hist["CustomerFeedback"]
neg = fb.str.contains(NEGATIVE, case=False)
leak = fb.str.contains(CANCEL_BROAD, case=False)
print("Não churners com texto negativo:", (neg & (y == 0)).sum(), "de", (y == 0).sum())
considering = fb.str.contains(r"considering (?:switching|leaving|other|cancel)|thinking (?:about|of) (?:switching|leaving|cancel)"
                              r"|may (?:switch|leave|cancel)|might (?:switch|leave|cancel)", case=False)
print("'Pensando em sair' × Churn:\n", pd.crosstab(considering, hist["Churn"]))
fi = inf["CustomerFeedback"]
print("\nInference — texto de cancelamento:", fi.str.contains(CANCEL_BROAD, case=False).sum(),
      "| negativo sem cancelamento:", (fi.str.contains(NEGATIVE, case=False) & ~fi.str.contains(CANCEL_BROAD, case=False)).sum(),
      "de", len(inf))

Não churners com texto negativo: 129 de 4130
'Pensando em sair' × Churn:
 Churn               No   Yes
CustomerFeedback            
False             4130  1499
True                 0     5



Inference — texto de cancelamento: 360 | negativo sem cancelamento: 40 de 1409


**Resposta:** Quase nunca. Só 129 dos 4.130 não churners têm texto negativo, e "pensando em sair" aparece só em 5 textos, todos de churners. No inference, só 40 textos são negativos sem falar em cancelamento. **O conteúdo do feedback dos churners indica que foi escrito depois da decisão de sair, mas a base não tem a data do texto: falta a TS confirmar como e quando ele é coletado.** Para clientes ativos, o texto quase não traz sinal antecipado.

## H. Desenho do modelo

### 30. Alguma variável vaza o alvo?

In [31]:
leak = hist["CustomerFeedback"].str.contains(CANCEL_BROAD, case=False)
pd.crosstab(leak.rename("texto diz que cancelou"), hist["Churn"])

Churn,No,Yes
texto diz que cancelou,,
False,4130,29
True,0,1475


**Resposta:** **Sim, o `CustomerFeedback`.** Os textos que dizem que o cliente já cancelou cobrem 1.475 churners e **nenhum** não churner, ou seja, 98% dos churns com zero falsos positivos. No inference, 25,6% dos textos têm esse padrão.

### 31. Quanto o modelo ganha com e sem o texto? (o que estaria disponível no momento de agir)

In [32]:
CATS = ["gender", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService"] + ADDONS + \
       ["Contract", "PaperlessBilling", "PaymentMethod"]

def structured_features(d):
    X = pd.get_dummies(d[CATS], drop_first=True).astype(int)
    X["SeniorCitizen"] = d["SeniorCitizen"]
    X["tenure"] = d["tenure_num"].fillna(d["tenure_num"].median())
    X["MonthlyCharges"] = d["MonthlyCharges"]
    X["TotalCharges"] = d["TotalCharges_imp"].fillna(0)
    X["MonthlyIncome"] = d["MonthlyIncome"]
    X["price_residual"] = d["price_residual"]
    return X

X_struct = structured_features(hist)
X_text = X_struct.assign(
    cancel_text=hist["CustomerFeedback"].str.contains(CANCEL_BROAD, case=False).astype(int),
    negative_text=hist["CustomerFeedback"].str.contains(NEGATIVE, case=False).astype(int),
)
models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "gbm": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=42),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for feat_name, X in [("só cadastrais", X_struct), ("cadastrais + texto", X_text),
                     ("cadastrais sem MonthlyIncome", X_struct.drop(columns="MonthlyIncome"))]:
    for model_name, m in models.items():
        s = cross_validate(m, X, y, cv=cv, scoring=["roc_auc", "f1", "accuracy"])
        rows.append({"features": feat_name, "modelo": model_name, "AUC": s["test_roc_auc"].mean(),
                     "F1": s["test_f1"].mean(), "acurácia": s["test_accuracy"].mean()})
pd.DataFrame(rows).round(4)

,features,modelo,AUC,F1,acurácia
0,só cadastrais,logreg,0.8398,0.5946,0.8009
1,só cadastrais,gbm,0.8276,0.5744,0.7943
2,cadastrais + texto,logreg,0.9996,0.9903,0.9949
3,cadastrais + texto,gbm,0.9997,0.9916,0.9956
4,cadastrais sem MonthlyIncome,logreg,0.8401,0.5934,0.8003
5,cadastrais sem MonthlyIncome,gbm,0.8271,0.5721,0.7934


**Resposta:** **Só com dados cadastrais:** regressão logística com AUC ≈ 0,84, F1 ≈ 0,59 e acurácia ≈ 80%. **Com indicadores do texto:** AUC ≈ 0,9996, F1 ≈ 0,99 e acurácia ≈ 99,5%. Tirar `MonthlyIncome` não muda nada. Daí a proposta de dois modelos: **(a) submissão** com texto e **(b) uso real** só com dados cadastrais, que é o que dá para usar antes de o cliente cancelar.

### 32. Há variáveis redundantes?

In [33]:
print("corr(tenure, TotalCharges):", round(hist["tenure_num"].corr(hist["TotalCharges_imp"]), 3))
print("corr(MonthlyCharges, price_residual):", round(hist["MonthlyCharges"].corr(hist["price_residual"]), 3))

corr(tenure, TotalCharges): 0.825
corr(MonthlyCharges, price_residual): 0.034


**Resposta:** Sim. `tenure` e `TotalCharges` têm correlação de 0,83, e `TotalCharges` é praticamente `Monthly × tenure`. `MonthlyCharges` é a soma dos serviços (pergunta 11). Para modelos de árvore isso não atrapalha, mas na regressão logística causa colinearidade. O `price_residual` é quase independente e não ajuda (pergunta 25).

## I. Valor financeiro

### 33. Quanto da receita mensal está em risco e em que segmentos?

In [34]:
rev = hist["MonthlyCharges"]
print(f"Receita mensal total: R$ {rev.sum():,.0f} | perdida com churners: R$ {rev[y == 1].sum():,.0f} "
      f"({rev[y == 1].sum() / rev.sum():.1%})")
print(f"Mensalidade média — churners: R$ {rev[y == 1].mean():.2f} | não churners: R$ {rev[y == 0].mean():.2f}")
for c in ["Contract", "InternetService", "PaymentMethod"]:
    lost = rev[y == 1].groupby(hist.loc[y == 1, c]).sum()
    print(f"% da receita perdida por {c}:", (lost / lost.sum()).round(3).to_dict())

Receita mensal total: R$ 364,010 | perdida com churners: R$ 112,243 (30.8%)
Mensalidade média — churners: R$ 74.63 | não churners: R$ 60.96
% da receita perdida por Contract: {'Month-to-month': 0.863, 'One year': 0.103, 'Two year': 0.034}
% da receita perdida por InternetService: {'DSL': 0.163, 'Fiber optic': 0.821, 'No': 0.016}
% da receita perdida por PaymentMethod: {'Bank transfer (automatic)': 0.148, 'Credit card (automatic)': 0.129, 'Electronic check': 0.601, 'Mailed check': 0.122}


**Resposta:** Os churners são 26,7% dos clientes, mas **30,8% da receita mensal** (R$ 112 mil de R$ 364 mil), porque pagam mais: R$ 74,63 contra R$ 60,96. A receita perdida vem **86% do contrato mensal, 82% de fibra e 60% de cheque eletrônico**.

### 34. A partir de que probabilidade de churn vale a pena oferecer uma ação de retenção?

**Resposta:** Depende do custo da ação e da taxa de aceite, que **ainda precisamos pedir à TS**. A regra é agir quando `p(churn) × taxa_de_aceite × valor_do_cliente > custo_da_ação`, em que o valor do cliente é a mensalidade × os meses de vida esperados.

### 35. Quais clientes valem mais a pena salvar?

In [35]:
print(pd.crosstab(pd.cut(hist["MonthlyCharges"], [0, 35, 70, 90, 200]), hist["Churn"], margins=True))
print("\nTaxa de churn por faixa:", y.groupby(pd.cut(hist["MonthlyCharges"], [0, 35, 70, 90, 200])).mean().round(3).to_dict())
print(f"Tenure médio — churners: {hist.loc[y == 1, 'tenure_num'].mean():.1f} | não churners: {hist.loc[y == 0, 'tenure_num'].mean():.1f}")

Churn             No   Yes   All
MonthlyCharges                  
(0, 35]         1250   151  1401
(35, 70]        1039   327  1366
(70, 90]         927   560  1487
(90, 200]        914   466  1380
All             4130  1504  5634

Taxa de churn por faixa: {Interval(0, 35, closed='right'): 0.108, Interval(35, 70, closed='right'): 0.239, Interval(70, 90, closed='right'): 0.377, Interval(90, 200, closed='right'): 0.338}
Tenure médio — churners: 18.4 | não churners: 37.4


**Resposta:** A faixa de R$ 70–90 tem 37,7% de churn e a acima de R$ 90 tem 33,8%. Os churners saem em média com 18,4 meses, contra 37,4 dos que ficam. **Os clientes de fibra no começo do contrato são o grupo de maior valor em risco.**